In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload;
# To disable autoreload; run %autoreload 0

In [0]:
# pip install -r requirements.txt
# %pip uninstall -y psycopg-binary
# %pip install "psycopg[pool]>=3.1.0"
dbutils.library.restartPython()

In [0]:
import os
from pprint import pprint

# openrouteservice API = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjUyYjgyNjE2YTVlZjQ3MGE5MDc3MDNlYTkwYTA4MWIxIiwiaCI6Im11cm11cjY0In0='

### destionation_api.py

##### 1. Import DestinationAPI class and client connection

In [0]:

os.environ["GEOAPIFY_API_KEY"] = '649d5b3a92ed4bbfa97f8c40906640f4'
from destination_api import DestinationAPI
client_geo = DestinationAPI(os.environ["GEOAPIFY_API_KEY"])

##### 2. Test get_places_radius()

In [0]:
features = client_geo.get_places_radius(location = "Seattle, WA"
                                        ,radius_m=3000
                                        ,categories="entertainment.museum"
                                        ,limit=3
                                    )
pprint(features)

##### 3. Test get_place_details()

In [0]:
place_id = features[0]["properties"]["place_id"]
place_id

In [0]:
detail = client_geo.get_place_details(place_id)
pprint(detail)

### destionation.py

In [0]:
os.environ["PGHOST"] = "ep-polished-silence-d8t2cf3b.database.us-east-2.cloud.databricks.com"
os.environ["LAKEBASE_ENDPOINT"] = "projects/weather-intelligence/branches/production/endpoints/primary"
os.environ["PGUSER"] = "tuvu.uwyo@gmail.com"


##### 1. Test categorize()

In [0]:
from destination import categorize

In [0]:
categories = ["entertainment", "entertainment.museum", "building", "building.tourism"]

result = categorize(categories)
print(result)

In [0]:
matches = [
    CATEGORY_MAP[c]
    for c in sorted(categories, key=lambda c: c.count("."), reverse=True)
    if c in CATEGORY_MAP
]
matches

In [0]:
matches[0]

##### 2. Test normalize_place()

In [0]:
from destination import normalize_place

doc = normalize_place(feature = features[0], detail = detail, location = "Seattle, WA")
pprint(doc)

##### 3. Test fetch_destionations()

In [0]:
from destination import fetch_destinations

docs = fetch_destinations(
    "Seattle, WA"
    ,api_key=os.environ["GEOAPIFY_API_KEY"]
    ,limit=5
    ,fetch_details=False
)

print(len(docs))
pprint(docs[0])

##### 4. Test upsert_documents()

In [0]:
from destination import upsert_documents

In [0]:
count = upsert_documents(docs)
print(count)

### embeddings.py

##### 1. Test embed_unembedded_documents()

In [0]:
# %pip install sentence-transformers


In [0]:
from embeddings import embed_unembedded_documents

In [0]:
import lakebase
count_embedded = embed_unembedded_documents(    
    lakebase.get_connection
    ,documents_table="destination_documents"
    ,embeddings_table="destination_embeddings"
)
print(count_embedded)

### search.py

##### 1. Test search_destination_documents()

In [0]:
from search import search_destination_documents

In [0]:
results = search_destination_documents("outdoor viewpoint with a view", lakebase.get_connection, top_k=3)
pprint(results)